<a href="https://colab.research.google.com/github/kalpanasahithi1-png/secure-sync/blob/main/Secure_sync_ipydo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# WEB PAGE SIMILARITY USING SVD + EIGENVALUE
# ============================================

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------
# 1. INPUT: 5 WEB PAGES
# ------------------------------------------------

pages = {
    "Page 1": """
    Cyber security protects computers networks and data from cyber attacks.
    Network security prevents unauthorized access and protects information.
    """,

    "Page 2": """
    Cyber security includes network security data protection and threat detection.
    Security professionals protect computers networks and information from attacks.
    """,

    "Page 3": """
    Machine learning uses data algorithms and mathematical models.
    Artificial intelligence and machine learning are used in many applications.
    """,

    "Page 4": """
    Search engines analyze web pages and rank relevant information.
    Search engines use keywords content similarity and page importance.
    """,

    "Page 5": """
    Data science uses statistics machine learning and data analysis.
    Machine learning models find patterns in large datasets.
    """
}


page_names = list(pages.keys())
page_text = list(pages.values())


# ------------------------------------------------
# 2. CONVERT WORDS INTO NUMERICAL VECTORS
# ------------------------------------------------

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(page_text)

print("======================================")
print("TF-IDF MATRIX")
print("======================================")

print("Number of pages:", X.shape[0])
print("Number of words:", X.shape[1])

print("\nTF-IDF Matrix:")
print(pd.DataFrame(
    X.toarray(),
    index=page_names
).round(3))


# ------------------------------------------------
# 3. COSINE SIMILARITY MATRIX
# ------------------------------------------------

similarity_matrix = cosine_similarity(X)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=page_names,
    columns=page_names
)

print("\n======================================")
print("COSINE SIMILARITY MATRIX")
print("======================================")

print(similarity_df.round(3))


# ------------------------------------------------
# 4. EIGENVALUE CENTRALITY
# ------------------------------------------------

# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(similarity_matrix)

# Find largest eigenvalue
largest_index = np.argmax(eigenvalues)

largest_eigenvalue = eigenvalues[largest_index]

# Corresponding eigenvector
importance_vector = np.abs(
    eigenvectors[:, largest_index]
)

# Normalize importance scores
eigen_scores = (
    importance_vector /
    np.sum(importance_vector)
)


# Create ranking table
eigen_df = pd.DataFrame({
    "Page": page_names,
    "Eigenvalue Score": eigen_scores
})

eigen_df = eigen_df.sort_values(
    "Eigenvalue Score",
    ascending=False
)

eigen_df["Eigen Rank"] = range(
    1,
    len(eigen_df) + 1
)


print("\n======================================")
print("EIGENVALUE CENTRALITY")
print("======================================")

print("Largest Eigenvalue:",
      round(largest_eigenvalue, 4))

print("\nEigenvalue Ranking:")

print(
    eigen_df[
        ["Eigen Rank", "Page", "Eigenvalue Score"]
    ].round(4).to_string(index=False)
)


# ------------------------------------------------
# 5. SVD
# ------------------------------------------------

# Convert TF-IDF matrix to normal NumPy matrix
X_dense = X.toarray()

# Singular Value Decomposition
U, singular_values, VT = np.linalg.svd(
    X_dense,
    full_matrices=False
)


print("\n======================================")
print("SVD")
print("======================================")

print("Singular Values:")

for i, value in enumerate(singular_values):
    print(
        "Topic",
        i + 1,
        ":",
        round(value, 4)
    )


# ------------------------------------------------
# 6. SVD PAGE-TOPIC SCORES
# ------------------------------------------------

# U contains page-topic relationships
# Multiply U by singular values

page_topic_scores = U * singular_values


print("\nPage-Topic Matrix:")

topic_columns = [
    "Topic " + str(i + 1)
    for i in range(len(singular_values))
]

page_topic_df = pd.DataFrame(
    page_topic_scores,
    index=page_names,
    columns=topic_columns
)

print(page_topic_df.round(3))


# ------------------------------------------------
# 7. SVD PAGE IMPORTANCE SCORE
# ------------------------------------------------

# Overall strength of each page
svd_scores = np.linalg.norm(
    page_topic_scores,
    axis=1
)

# Normalize
svd_scores = (
    svd_scores /
    np.sum(svd_scores)
)


svd_df = pd.DataFrame({
    "Page": page_names,
    "SVD Score": svd_scores
})

svd_df = svd_df.sort_values(
    "SVD Score",
    ascending=False
)

svd_df["SVD Rank"] = range(
    1,
    len(svd_df) + 1
)


print("\n======================================")
print("SVD PAGE RANKING")
print("======================================")

print(
    svd_df[
        ["SVD Rank", "Page", "SVD Score"]
    ].round(4).to_string(index=False)
)


# ------------------------------------------------
# 8. COMPARE BOTH RANKINGS
# ------------------------------------------------

comparison = pd.merge(
    eigen_df[
        ["Page", "Eigen Rank", "Eigenvalue Score"]
    ],
    svd_df[
        ["Page", "SVD Rank", "SVD Score"]
    ],
    on="Page"
)

comparison = comparison.sort_values(
    "Eigen Rank"
)


print("\n======================================")
print("FINAL COMPARISON")
print("======================================")

print(
    comparison.round(4).to_string(index=False)
)


# ------------------------------------------------
# 9. FINAL RESULT
# ------------------------------------------------

best_eigen_page = eigen_df.iloc[0]["Page"]
best_svd_page = svd_df.iloc[0]["Page"]

print("\n======================================")
print("FINAL RESULT")
print("======================================")

print(
    "Most important page using Eigenvalue:",
    best_eigen_page
)

print(
    "Most important page using SVD:",
    best_svd_page
)

print("\nProject completed successfully!")

TF-IDF MATRIX
Number of pages: 5
Number of words: 47

TF-IDF Matrix:
           0      1      2      3      4      5      6      7      8      9   \
Page 1  0.253  0.000  0.000  0.000  0.000  0.000  0.204  0.204  0.000  0.409   
Page 2  0.000  0.000  0.000  0.000  0.000  0.000  0.202  0.202  0.000  0.202   
Page 3  0.000  0.279  0.000  0.000  0.279  0.279  0.000  0.000  0.000  0.000   
Page 4  0.000  0.000  0.000  0.227  0.000  0.000  0.000  0.000  0.227  0.000   
Page 5  0.000  0.000  0.269  0.000  0.000  0.000  0.000  0.000  0.000  0.000   

        ...     37     38     39     40     41     42     43     44     45  \
Page 1  ...  0.000  0.409  0.000  0.000  0.000  0.253  0.000  0.000  0.000   
Page 2  ...  0.000  0.607  0.000  0.000  0.251  0.000  0.000  0.000  0.000   
Page 3  ...  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.279  0.225   
Page 4  ...  0.454  0.000  0.227  0.000  0.000  0.000  0.227  0.000  0.000   
Page 5  ...  0.000  0.000  0.000  0.269  0.000  0.000  0.000

In [ ]:
# ==========================================================
# F5: WEB PAGE SIMILARITY VIA SVD + EIGENVALUE
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ==========================================================
# STEP 1: FIVE WEB PAGES
# ==========================================================

web_pages = [
    """
    Cyber security protects computers networks and data.
    Cyber attacks can steal information and damage systems.
    Network security helps prevent unauthorized access.
    """,

    """
    Network security protects computer networks from attacks.
    Cyber security includes data protection threat detection
    and secure communication between computers.
    """,

    """
    Machine learning is a branch of artificial intelligence.
    Machine learning uses data algorithms and mathematical
    models to find patterns and make predictions.
    """,

    """
    Search engines analyze web pages and keywords.
    Search engines rank pages according to relevance,
    similarity and importance of information.
    """,

    """
    Data science combines statistics programming and machine
    learning. Data analysis helps discover useful patterns
    from large datasets.
    """
]

page_names = [
    "Page 1",
    "Page 2",
    "Page 3",
    "Page 4",
    "Page 5"
]


# ==========================================================
# STEP 2: CONVERT TEXT INTO TF-IDF VECTORS
# ==========================================================

vectorizer = TfidfVectorizer(
    stop_words="english"
)

X = vectorizer.fit_transform(web_pages)

X = X.toarray()

print("\nTF-IDF MATRIX")
print("=" * 60)

tfidf_table = pd.DataFrame(
    X,
    index=page_names
)

print(tfidf_table.round(3))


# ==========================================================
# STEP 3: COSINE SIMILARITY
# ==========================================================

S = cosine_similarity(X)

similarity_table = pd.DataFrame(
    S,
    index=page_names,
    columns=page_names
)

print("\n\nCOSINE SIMILARITY MATRIX")
print("=" * 60)

print(similarity_table.round(3))


# ==========================================================
# STEP 4: EIGENVALUE ANALYSIS
# ==========================================================

eigenvalues, eigenvectors = np.linalg.eigh(S)

print("\n\nEIGENVALUES")
print("=" * 60)

for i, value in enumerate(eigenvalues):
    print(
        f"Eigenvalue {i + 1}: {value:.4f}"
    )


# Largest eigenvalue
index = np.argmax(eigenvalues)

largest_eigenvalue = eigenvalues[index]

# Corresponding eigenvector
centrality = np.abs(
    eigenvectors[:, index]
)

# Normalize
centrality = centrality / np.sum(centrality)


# Create ranking
eigen_result = pd.DataFrame({
    "Page": page_names,
    "Eigen Score": centrality
})

eigen_result = eigen_result.sort_values(
    "Eigen Score",
    ascending=False
)

eigen_result["Rank"] = range(
    1,
    len(eigen_result) + 1
)

print("\nEIGENVALUE PAGE RANKING")
print("=" * 60)

print(
    eigen_result[
        ["Rank", "Page", "Eigen Score"]
    ].round(4).to_string(index=False)
)


# ==========================================================
# STEP 5: SVD
# ==========================================================

U, sigma, VT = np.linalg.svd(
    X,
    full_matrices=False
)

print("\n\nSINGULAR VALUES")
print("=" * 60)

for i, value in enumerate(sigma):
    print(
        f"Topic {i + 1}: {value:.4f}"
    )


# ==========================================================
# STEP 6: PAGE-TOPIC MATRIX
# ==========================================================

page_topic = U * sigma

topic_names = [
    f"Topic {i + 1}"
    for i in range(len(sigma))
]

topic_table = pd.DataFrame(
    page_topic,
    index=page_names,
    columns=topic_names
)

print("\n\nSVD PAGE-TOPIC MATRIX")
print("=" * 60)

print(topic_table.round(3))


# ==========================================================
# STEP 7: SVD PAGE SCORE
# ==========================================================

# Calculate total topic strength
svd_score = np.linalg.norm(
    page_topic,
    axis=1
)

# Normalize
svd_score = svd_score / np.sum(svd_score)


svd_result = pd.DataFrame({
    "Page": page_names,
    "SVD Score": svd_score
})

svd_result = svd_result.sort_values(
    "SVD Score",
    ascending=False
)

svd_result["Rank"] = range(
    1,
    len(svd_result) + 1
)

print("\nSVD PAGE RANKING")
print("=" * 60)

print(
    svd_result[
        ["Rank", "Page", "SVD Score"]
    ].round(4).to_string(index=False)
)


# ==========================================================
# STEP 8: FINAL COMPARISON
# ==========================================================

comparison = pd.merge(
    eigen_result[
        ["Page", "Rank", "Eigen Score"]
    ],
    svd_result[
        ["Page", "Rank", "SVD Score"]
    ],
    on="Page"
)

comparison.columns = [
    "Page",
    "Eigen Rank",
    "Eigen Score",
    "SVD Rank",
    "SVD Score"
]

comparison = comparison.sort_values(
    "Eigen Rank"
)

print("\n\nFINAL RANKING COMPARISON")
print("=" * 60)

print(
    comparison.round(4).to_string(index=False)
)


# ==========================================================
# STEP 9: BEST PAGE
# ==========================================================

best_eigen = eigen_result.iloc[0]["Page"]
best_svd = svd_result.iloc[0]["Page"]

print("\n\nFINAL RESULT")
print("=" * 60)

print(
    "Best page according to Eigenvalue:",
    best_eigen
)

print(
    "Best page according to SVD:",
    best_svd
)

print("\nEigenvalue and SVD rankings have been compared.")


TF-IDF MATRIX
           0     1      2      3     4      5      6      7      8      9   \
Page 1  0.249  0.00  0.000  0.000  0.00  0.000  0.201  0.000  0.000  0.000   
Page 2  0.000  0.00  0.000  0.000  0.00  0.000  0.217  0.000  0.000  0.269   
Page 3  0.000  0.00  0.257  0.000  0.00  0.257  0.000  0.257  0.000  0.000   
Page 4  0.000  0.22  0.000  0.000  0.22  0.000  0.000  0.000  0.000  0.000   
Page 5  0.000  0.00  0.000  0.279  0.00  0.000  0.000  0.000  0.279  0.000   

        ...     45    46     47     48     49     50     51     52     53  \
Page 1  ...  0.402  0.00  0.000  0.249  0.249  0.000  0.249  0.000  0.000   
Page 2  ...  0.434  0.00  0.000  0.000  0.000  0.269  0.000  0.000  0.000   
Page 3  ...  0.000  0.00  0.000  0.000  0.000  0.000  0.000  0.000  0.257   
Page 4  ...  0.000  0.22  0.000  0.000  0.000  0.000  0.000  0.000  0.000   
Page 5  ...  0.000  0.00  0.279  0.000  0.000  0.000  0.000  0.279  0.000   

          54  
Page 1  0.00  
Page 2  0.00  
Page 3  